# XAUUSD — Pronóstico de ruta: Monte Carlo Markov-Switching

**Qué hace este notebook:**
1. Descarga datos H1 reales de XAUUSD desde Yahoo Finance (o puedes subir tu CSV de MT5).
2. Calibra un HMM de 3 regímenes (alcista / bajista / rango).
3. Detecta el régimen actual del mercado.
4. Lanza 10,000 simulaciones Monte Carlo desde el precio actual.
5. Reporta la ruta pronosticada, niveles alcanzables y probabilidad de tocar un objetivo.
6. Dibuja el fan chart y el histograma del precio final.

**Cómo usarlo:** menú `Entorno de ejecución → Ejecutar todas` (Runtime → Run all).
Ajusta `CONFIG` y `TARGET` en la celda 2 si quieres cambiar horizonte u objetivo.

> ⚠️ Herramienta educativa/estadística. No es asesoría financiera: el modelo asume
> que el futuro se parece estadísticamente al pasado reciente, lo cual puede no cumplirse.


In [ ]:
# @title 1) Instalar dependencias
%pip install -q hmmlearn yfinance
print("Dependencias listas.")


In [ ]:
# @title 2) Configuración
CONFIG = {
    "n_states":         3,       # regímenes del HMM
    "n_paths":          10000,   # simulaciones Monte Carlo
    "horizon_hours":    48,      # horizonte del pronóstico (velas H1)
    "checkpoint_every": 4,       # puntos de control de la ruta (cada 4 horas)
    "confidence":       0.80,    # banda de confianza para la ruta
    "ou_kappa":         0.03,    # reversión a la media en régimen rango
    "seed":             42,
    "out_prefix":       "xauusd_ruta",
}

# Precio objetivo opcional: pon un número (ej. 4280.0) para calcular la
# probabilidad de que el precio lo TOQUE dentro del horizonte, o deja None.
TARGET = None

# Fuente de datos: "yahoo" descarga automáticamente; "csv" te pide subir
# un archivo exportado de MT5 (debe tener una columna CLOSE).
DATA_SOURCE = "yahoo"
YAHOO_PERIOD = "1y"   # cuánta historia H1 bajar: "3mo", "6mo", "1y", "2y" (máx. ~730 días)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("Configuración cargada.")


In [ ]:
# @title 3) Datos: Yahoo Finance (XAUUSD=X) o CSV de MT5
def load_yahoo(period="1y"):
    import yfinance as yf
    closes, used = None, None
    for ticker in ["XAUUSD=X", "GC=F"]:   # spot oro; futuros COMEX como respaldo
        try:
            df = yf.download(ticker, interval="1h", period=period,
                             progress=False, auto_adjust=True)
            if df is not None and len(df) > 500:
                closes = pd.to_numeric(df["Close"].squeeze(), errors="coerce").dropna().values
                used = ticker
                break
        except Exception as e:
            print(f"[DATOS] Fallo con {ticker}: {e}")
    if closes is None:
        raise RuntimeError("No pude descargar datos de Yahoo Finance. "
                           "Reintenta, o usa DATA_SOURCE='csv'.")
    print(f"[DATOS] {len(closes)} velas H1 de {used} descargadas de Yahoo Finance")
    print(f"[DATOS] Último cierre: {closes[-1]:.2f}")
    return closes


def load_mt5_csv_upload():
    from google.colab import files
    print("Sube tu CSV exportado de MT5 (XAUUSD H1, con columna CLOSE)...")
    uploaded = files.upload()
    path = list(uploaded.keys())[0]
    for sep in ["\t", ",", ";"]:
        try:
            df = pd.read_csv(path, sep=sep)
            if df.shape[1] >= 4:
                break
        except Exception:
            continue
    else:
        raise ValueError("No pude leer el CSV. Verifica el separador.")
    df.columns = [c.strip().strip("<>").upper() for c in df.columns]
    if "CLOSE" not in df.columns:
        raise ValueError(f"No encuentro la columna CLOSE. Columnas: {list(df.columns)}")
    closes = pd.to_numeric(df["CLOSE"], errors="coerce").dropna().values
    print(f"[DATOS] {len(closes)} velas H1 cargadas de {path}")
    print(f"[DATOS] Último cierre: {closes[-1]:.2f}")
    return closes


closes = load_yahoo(YAHOO_PERIOD) if DATA_SOURCE == "yahoo" else load_mt5_csv_upload()
returns = np.diff(np.log(closes))


In [ ]:
# @title 4) Calibración HMM (retorno + canal de drift para separar tendencias)
def fit_hmm(returns, n_states, seed):
    """Devuelve (transición, mus, sigmas, estados ocultos) ya ordenados:
       0=alcista, 1=bajista, 2=rango."""
    from hmmlearn.hmm import GaussianHMM
    SCALE = 100.0
    drift = pd.Series(returns).rolling(24, min_periods=1).mean().values
    X = np.column_stack([returns * SCALE, drift * SCALE * 20.0])
    best_model, best_ll = None, -np.inf
    for s in range(5):
        m = GaussianHMM(n_components=n_states, covariance_type="diag",
                        n_iter=500, random_state=seed + s, tol=1e-6,
                        min_covar=1e-6)
        m.fit(X)
        ll = m.score(X)
        if ll > best_ll:
            best_model, best_ll = m, ll
    model  = best_model
    trans  = model.transmat_
    mus    = model.means_[:, 0] / SCALE
    covs   = np.array([np.diag(c)[0] for c in model.covars_])
    sigmas = np.sqrt(covs) / SCALE
    order_key = model.means_[:, 1]
    hidden = model.predict(X)
    print(f"[HMM] hmmlearn OK | log-verosimilitud: {best_ll:.1f}")
    # reordenar: 0=alcista (drift máx), 1=bajista (drift mín), 2=rango
    order = [int(np.argmax(order_key)), int(np.argmin(order_key))]
    order.append([s for s in range(n_states) if s not in order][0])
    idx   = np.array(order)
    remap = {old: new for new, old in enumerate(idx)}
    return (trans[np.ix_(idx, idx)], mus[idx], sigmas[idx],
            np.array([remap[s] for s in hidden]))


def print_calibration(trans, mus, sigmas, hidden, current_state):
    names = ["ALCISTA", "BAJISTA", "RANGO  "]
    print("\n===== REGÍMENES CALIBRADOS =====")
    for s in range(3):
        occup = 100.0 * np.mean(hidden == s)
        dur   = 1.0 / max(1e-9, 1.0 - trans[s, s])
        print(f"  {names[s]} | mu={mus[s]*100:+.4f}%/H1  sigma={sigmas[s]*100:.4f}%  "
              f"ocupación={occup:5.1f}%  duración media={dur:6.1f}h (~{dur/24:.1f} días)")
    print(f"\n  >>> RÉGIMEN ACTUAL DETECTADO: {names[current_state].strip()} <<<")
    print(f"  (el pronóstico arranca desde este régimen, no desde cero)\n")


trans, mus, sigmas, hidden = fit_hmm(returns, CONFIG["n_states"], CONFIG["seed"])
current_state = int(hidden[-1])
print_calibration(trans, mus, sigmas, hidden, current_state)


In [ ]:
# @title 5) Monte Carlo Markov-switching
def simulate(trans, mus, sigmas, p0, s0, n_paths, n_bars, ou_kappa, seed):
    """Simula n_paths caminos de n_bars velas H1 desde precio p0 y régimen s0.
       Devuelve matriz (n_bars+1, n_paths); la fila 0 es el precio inicial."""
    rng    = np.random.default_rng(seed)
    states = np.full(n_paths, s0)
    logp   = np.full(n_paths, np.log(p0))
    anchor = logp.copy()
    cum_trans = np.cumsum(trans, axis=1)
    prices = np.empty((n_bars + 1, n_paths))
    prices[0] = p0
    for t in range(1, n_bars + 1):
        u = rng.random(n_paths)
        new_states = np.empty(n_paths, dtype=int)
        for s in range(3):
            mask = states == s
            if mask.any():
                new_states[mask] = np.searchsorted(cum_trans[s], u[mask])
        entering_range = (new_states == 2) & (states != 2)
        anchor[entering_range] = logp[entering_range]
        states = new_states
        z = rng.standard_normal(n_paths)
        trend = states != 2
        logp = np.where(
            trend,
            logp + mus[np.clip(states, 0, 1)] + sigmas[np.clip(states, 0, 1)] * z,
            logp + ou_kappa * (anchor - logp) + sigmas[2] * z,
        )
        prices[t] = np.exp(logp)
    return prices


p0 = float(closes[-1])
print(f"[MC] Simulando {CONFIG['n_paths']} caminos x {CONFIG['horizon_hours']} velas H1 "
      f"desde {p0:.2f}...")
paths = simulate(trans, mus, sigmas, p0, current_state,
                 CONFIG["n_paths"], CONFIG["horizon_hours"],
                 CONFIG["ou_kappa"], CONFIG["seed"])
print("[MC] Listo.")


In [ ]:
# @title 6) Reporte de la ruta
def report_route(paths, p0, cfg, target=None):
    conf   = cfg["confidence"]
    q_lo   = (1.0 - conf) / 2.0 * 100.0        # 80% -> percentil 10
    q_hi   = 100.0 - q_lo                      # 80% -> percentil 90
    step   = cfg["checkpoint_every"]
    n_bars = paths.shape[0] - 1

    print(f"===== RUTA PRONOSTICADA ({paths.shape[1]} simulaciones, "
          f"banda de confianza {conf*100:.0f}%) =====")
    print(f"  Inicio: precio XAUUSD {p0:.2f}\n")
    print(f"  {'hora':>6} | {'mediana':>9} | {'banda ' + format(conf*100,'.0f') + '%':^23} | {'P(subida)':>9}")
    print(f"  {'-'*6} | {'-'*9} | {'-'*23} | {'-'*9}")
    for h in range(step, n_bars + 1, step):
        med = np.median(paths[h])
        lo  = np.percentile(paths[h], q_lo)
        hi  = np.percentile(paths[h], q_hi)
        pup = 100.0 * np.mean(paths[h] > p0)
        print(f"  En {h:2d}h | {med:9.2f} | [{lo:9.2f} - {hi:9.2f}] | {pup:7.1f}%")

    # ---- niveles alcanzables (primer paso: usa el MÁXIMO/MÍNIMO del camino)
    run_max = paths.max(axis=0)
    run_min = paths.min(axis=0)
    lvl_up   = np.percentile(run_max, (1.0 - conf) * 100.0)  # 80% de caminos lo tocan
    lvl_down = np.percentile(run_min, conf * 100.0)
    lvl_up50   = np.percentile(run_max, 50.0)
    lvl_down50 = np.percentile(run_min, 50.0)
    print(f"\n===== NIVELES ALCANZABLES EN {n_bars}h (primer paso) =====")
    print(f"  Con {conf*100:.0f}% de probabilidad el precio TOCA >= {lvl_up:.2f} "
          f"({(lvl_up-p0)/0.10:+.0f} pips)")
    print(f"  Con {conf*100:.0f}% de probabilidad el precio TOCA <= {lvl_down:.2f} "
          f"({(lvl_down-p0)/0.10:+.0f} pips)")
    print(f"  Con 50% de probabilidad el precio TOCA >= {lvl_up50:.2f} "
          f"({(lvl_up50-p0)/0.10:+.0f} pips)")
    print(f"  Con 50% de probabilidad el precio TOCA <= {lvl_down50:.2f} "
          f"({(lvl_down50-p0)/0.10:+.0f} pips)")

    # ---- probabilidad de tocar un objetivo dado por el usuario
    if target is not None:
        if target > p0:
            p_touch = 100.0 * np.mean(run_max >= target)
            touch_hours = [np.argmax(paths[:, j] >= target)
                           for j in range(paths.shape[1]) if run_max[j] >= target]
        else:
            p_touch = 100.0 * np.mean(run_min <= target)
            touch_hours = [np.argmax(paths[:, j] <= target)
                           for j in range(paths.shape[1]) if run_min[j] <= target]
        print(f"\n===== OBJETIVO {target:.2f} =====")
        print(f"  Probabilidad de TOCARLO en {n_bars}h: {p_touch:.1f}%")
        if touch_hours:
            print(f"  Si lo toca, hora típica (mediana): {int(np.median(touch_hours))}h "
                  f"| más temprano (p10): {int(np.percentile(touch_hours,10))}h "
                  f"| más tarde (p90): {int(np.percentile(touch_hours,90))}h")


report_route(paths, p0, CONFIG, TARGET)


In [ ]:
# @title 7) Gráficos: fan chart e histograma final
def make_plots(paths, p0, cfg, target=None):
    pre    = cfg["out_prefix"]
    n_bars = paths.shape[0] - 1
    hours  = np.arange(n_bars + 1)

    # (a) fan chart
    fig, ax = plt.subplots(figsize=(12, 5.5))
    for lo, hi, a in [(2.5, 97.5, 0.15), (10, 90, 0.25), (25, 75, 0.35)]:
        ax.fill_between(hours, np.percentile(paths, lo, axis=1),
                        np.percentile(paths, hi, axis=1),
                        color="#1565c0", alpha=a,
                        label=f"banda {hi-lo:.0f}%")
    ax.plot(hours, np.median(paths, axis=1), "k-", lw=2, label="mediana (ruta central)")
    rng = np.random.default_rng(1)
    for j in rng.choice(paths.shape[1], size=min(25, paths.shape[1]), replace=False):
        ax.plot(hours, paths[:, j], lw=0.4, alpha=0.4, color="#616161")
    ax.axhline(p0, color="k", ls=":", lw=1)
    if target is not None:
        ax.axhline(target, color="#c62828", ls="--", lw=1.5, label=f"objetivo {target:.2f}")
    ax.set_title(f"XAUUSD: ruta pronosticada a {n_bars}h "
                 f"({paths.shape[1]} simulaciones Markov-switching)")
    ax.set_xlabel("horas"); ax.set_ylabel("USD/oz"); ax.legend(loc="upper left")
    fig.tight_layout(); fig.savefig(f"{pre}_fanchart.png", dpi=120); plt.show()

    # (b) histograma del precio final
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.hist(paths[-1], bins=120, color="#2e7d32", alpha=0.85)
    ax.axvline(p0, color="k", ls=":", lw=1.5, label=f"inicio {p0:.2f}")
    ax.axvline(np.median(paths[-1]), color="#c62828", lw=1.5,
               label=f"mediana {np.median(paths[-1]):.2f}")
    if target is not None:
        ax.axvline(target, color="#ef6c00", ls="--", lw=1.5, label=f"objetivo {target:.2f}")
    ax.set_title(f"Distribución del precio a {n_bars}h")
    ax.set_xlabel("USD/oz"); ax.legend()
    fig.tight_layout(); fig.savefig(f"{pre}_final.png", dpi=120); plt.show()
    print(f"[GRÁFICOS] También guardados como {pre}_fanchart.png y {pre}_final.png "
          f"(panel de archivos, a la izquierda).")


make_plots(paths, p0, CONFIG, TARGET)
print("\n[FIN] Ajusta CONFIG y TARGET en la celda 2 y vuelve a ejecutar todo.")
